In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scipy as sp
import copy
from IPython.display import Image
from qutip import *
import time
import math

In [ ]:
dim = 2      # number of levels for each qubit
GHz = 1e8    #1e9 (0.1GHz)
ns = 1e-9
b = destroy(dim)                     # sigma minus operator

s0m = tensor(b, qeye(dim))           # sigma minus on 0
s0p = tensor(b.dag(), qeye(dim))     # sigma plus on 0
s1m = tensor(qeye(dim), b)           # sigma minus on 1
s1p = tensor(qeye(dim), b.dag())     # sigma plus on 1

omega = [2*np.pi*4.2*GHz, 2*np.pi*4.7*GHz]            # qubit transition frequencies [rad/s]
Delta = [2*np.pi*0*GHz,2*np.pi*0*GHz] #[-.33, -.33]   # anharmonicities [rad/s]
Omega = [2*np.pi*1*GHz, 2*np.pi*1.1*GHz]              # control-qubit drive strength [rad/s]
J = 2*np.pi*0*GHz                                     # passive interaction [rad/s]

c_max = 10  # upper bound for control amplitudes
c_min = 0   # lower bound for control amplitudes
T = 10*ns   # target time
L = 2     # for Gaussian, always set to 2

H0 = ( omega[0]*s0p*s0m + omega[1]*s1p*s1m ) + \
     (1/2)*( Delta[0]*s0p*s0p*s0m*s0m + Delta[1]*s1p*s1p*s1m*s1m ) + \
     J*(s0p*s1m+s0m*s1p)

Hc0m = Omega[0]*(s0m)
Hc1m = Omega[1]*(s1m)
Hc0p = Omega[0]*(s0p)
Hc1p = Omega[1]*(s1p)

'''
# Piecewise Constant Control on qubit 0
def C0(t, args):
    c = args['c']     # list of control parameters [c11,...,c1L]
    T = args['T']     # target time 
    L = args['L']     # L pieces
    t_inv = list(np.linspace(0,T,L+1))     # endpoints of time interval
    return( np.sum([c[0][i] * ( float(t>=t_inv[i] and t<t_inv[i+1]) ) for i in range(L)])+c[0][L-1]*float(t==t_inv[L])  )

def C1(t, args):
    c = args['c']     # list of control parameters [c21,...,c2L]
    T = args['T']     # target time 
    L = args['L']     # L pieces
    t_inv = list(np.linspace(0,T,L+1))     # endpoints of time interval
    return( np.sum([c[1][i] * ( float(t>=t_inv[i] and t<t_inv[i+1]) ) for i in range(L)])+c[1][L-1]*float(t==t_inv[L])  )

# Piecewise Constant Control on qubit 1
def C2(t, args):
    c = args['c']     # list of control parameters [c31,...,c3L]
    T = args['T']     # target time 
    L = args['L']     # L pieces
    t_inv = list(np.linspace(0,T,L+1))     # endpoints of time interval
    return( np.sum([c[2][i] * ( float(t>=t_inv[i] and t<t_inv[i+1]) ) for i in range(L)])+c[2][L-1]*float(t==t_inv[L])  )

def C3(t, args):
    c = args['c']     # list of control parameters [c41,...,c4L]
    T = args['T']     # target time 
    L = args['L']     # L pieces
    t_inv = list(np.linspace(0,T,L+1))     # endpoints of time interval
    return( np.sum([c[3][i] * ( float(t>=t_inv[i] and t<t_inv[i+1]) ) for i in range(L)])+c[3][L-1]*float(t==t_inv[L])  )
'''

# Gaussian Control on qubit 0
def C0(t, args):
    c = args['c']     # list of control parameters [c11, c12]
    T = args['T']     # target time
    return( c[0][0] * math.exp(-(t-(T/2))**2/((c[0][1])**2) * ( float(t>=0 and t<=T) ) ))
           
def C1(t, args):
    c = args['c']     # list of control parameters [c21, c22]
    T = args['T']     # target time
    return( c[1][0] * math.exp(-(t-(T/2))**2/((c[1][1])**2) * ( float(t>=0 and t<=T) ) ))

# Gaussian Control on qubit 1
def C2(t, args):
    c = args['c']     # list of control parameters [c11, c12]
    T = args['T']     # target time
    return( c[2][0] * math.exp(-(t-(T/2))**2/((c[2][1])**2) * ( float(t>=0 and t<=T) ) ))
           
def C3(t, args):
    c = args['c']     # list of control parameters [c21, c22]
    T = args['T']     # target time
    return( c[3][0] * math.exp(-(t-(T/2))**2/((c[3][1])**2) * ( float(t>=0 and t<=T) ) ))     
           

# Marker
def ep0(t, args):
    omega = args['omega']
    return np.exp(1j *omega[0]*t)

def em0(t, args):
    omega = args['omega']
    return np.exp(-1j *omega[0]*t)

def ep1(t, args):
    omega = args['omega']
    return np.exp(1j *omega[1]*t)

def em1(t, args):
    omega = args['omega']
    return np.exp(-1j *omega[1]*t)

# Control Functions C_i_j_k: i=0,1; j=m,p related to sm/sp; k=m,p related to (sm-sp) or (sm+sp)
def C0mm(t, args):
    return 1j*C0(t, args)*ep0(t, args)
def C0pm(t, args):
    return -1j*C0(t, args)*em0(t, args)

def C0mp(t, args):
    return C1(t, args)*ep0(t, args)
def C0pp(t, args):
    return C1(t, args)*em0(t, args)

def C1mm(t, args):
    return 1j*C2(t, args)*ep1(t, args)
def C1pm(t, args):
    return -1j*C2(t, args)*em1(t, args)

def C1mp(t, args):
    return C3(t, args)*ep1(t, args)
def C1pp(t, args):
    return C3(t, args)*em1(t, args)

# Total Hamiltonain
H = [H0, [Hc0m,C0mm], [Hc0p,C0pm], [Hc0m,C0mp], [Hc0p,C0pp], [Hc1m,C1mm], [Hc1p,C1pm], [Hc1m,C1mp], [Hc1p,C1pp]]   

In [ ]:
GA = sigmax() # Target Unitary on qubit 0
GB = sigmax() # Target Unitary on qubit 1

In [ ]:
def OBJ(c,T,L,omega,H,GA,GB):
    
    args = {
        'c': c,
        'T': T,
        'L': L,
        'omega': omega
    }
    
    U = propagator(H, T, args=args,options=Options(nsteps=1700))
    
    # SVD for partial traces
    sigma = np.sqrt((((tensor(GA,qeye(dim)).dag()*U).ptrace(1))*((tensor(GA,qeye(dim)).dag()*U).ptrace(1)).dag()).eigenstates()[0])
    lambd = np.sqrt((((tensor(qeye(dim),GB).dag()*U).ptrace(0))*((tensor(qeye(dim),GB).dag()*U).ptrace(0)).dag()).eigenstates()[0])
    
    # sum of fidelity
    return sum(sigma)/(2*dim**2)+sum(lambd)/(2*dim**2)

In [ ]:
# GRadient Ascent Pulse Engineering (GRAPE)

def GRAPE(T,L,omega,H,GA,GB,c=None):
    eps = 0.01               # ascending rate
    dx = 0.01                # gradient increment
    threshold = 0.000001    # if cost changes less than threshold, then halt
    itera_max = 500          # max iteration number
    count = 0               # counting iterations
    diff_cost = np.inf      # initial cost difference
    d = T/L                 # time interval length
    record_cost = []
    factor = 0.5
    # Step1-1, Guess initial
    if c is None:
        c = [[float(np.random.uniform(c_min,c_max)) for _ in range(L)],    # for C1
             [float(np.random.uniform(c_min,c_max)) for _ in range(L)],    # for C2
             [float(np.random.uniform(c_min,c_max)) for _ in range(L)],    # for C3
             [float(np.random.uniform(c_min,c_max)) for _ in range(L)]]    # for C4
    # Step1-2, Compute initial cost
    cost = OBJ(c,T,L,omega,H,GA,GB)
    print(cost)
    record_cost.append(cost)
    # Start while loop
    while (count<=itera_max) and (diff_cost>threshold):
        #print(count)
        count += 1
        # Step 2, compute derivative numerically by Df(x0) = (f(x0+dx)-f(x0)) / dx
        D_J_c = [ [ 0 for l in range(L)] for m in range(4)]
        for m in range(4):
            for l in range(L):
                c_tmp1 = copy.deepcopy(c)
                c_tmp2 = copy.deepcopy(c)
                c_tmp1[m][l] += dx/2
                c_tmp2[m][l] -= dx/2
                D_J_c[m][l] += (OBJ(c_tmp1,T,L,omega,H,GA,GB) - OBJ(c_tmp2,T,L,omega,H,GA,GB)) / dx
        # Step 3, normalization for gradient
        norm2_D = 0
        for m in range(4):
            for l in range(L):
                norm2_D += D_J_c[m][l]**2
        norm_D = np.sqrt(norm2_D)
        D_J_c = [[eps*D_J_c[m][l]/norm_D for l in range(L)] for m in range(4)]
        # Step4, gradient ascend
        c_old = np.array(c)
        c = np.array(c) + np.array(D_J_c)
        # Step5, Compute cost
        new_cost = OBJ(c,T,L,omega,H,GA,GB)
        if new_cost > record_cost[-1]:
            print(new_cost)
            diff_cost = abs(new_cost - record_cost[-1])
            record_cost.append(new_cost)
        else:
            c = c_old
            dx *= factor
            eps *= factor
    return record_cost, c, count
    

In [ ]:
# Original-style Gaussian GRAPE with amplitude/width scaling
# This keeps the structure of the original GRAPE cell above, but treats Gaussian width separately.

def GRAPE_Gaussian_scaled(T,L,omega,H,GA,GB,c=None):
    eps = 0.01                 # ascending rate, same role as original GRAPE
    dx_amp = 0.01              # finite-difference step for Gaussian amplitude
    dx_width = 0.01*ns         # finite-difference step for Gaussian width
    threshold = 0.000001       # if cost changes less than threshold, then halt
    itera_max = 500            # max iteration number
    count = 0                 # counting iterations
    diff_cost = np.inf        # initial cost difference
    record_cost = []
    factor = 0.5
    width_min = 0.05*T
    width_max = 1.50*T

    # Step1-1, Guess initial Gaussian parameters: [amplitude, width]
    if c is None:
        c = [[float(np.random.uniform(c_min,c_max)), float(np.random.uniform(0.20*T,1.00*T))] for _ in range(4)]
    else:
        c = copy.deepcopy(c)

    c = np.asarray(c, dtype=float)
    c[:,0] = np.clip(c[:,0], c_min, c_max)
    c[:,1] = np.clip(c[:,1], width_min, width_max)

    # Step1-2, Compute initial cost
    cost = OBJ(c.tolist(),T,L,omega,H,GA,GB)
    print(cost)
    record_cost.append(cost)

    # Start while loop
    while (count<=itera_max) and (diff_cost>threshold):
        count += 1

        # Step 2, compute derivative numerically by Df(x0) = (f(x0+dx)-f(x0)) / dx
        D_J_c = [ [ 0 for l in range(L)] for m in range(4)]
        for m in range(4):
            for l in range(L):
                local_dx = dx_amp if l == 0 else dx_width
                c_tmp1 = copy.deepcopy(c)
                c_tmp2 = copy.deepcopy(c)
                c_tmp1[m][l] += local_dx/2
                c_tmp2[m][l] -= local_dx/2
                c_tmp1[:,0] = np.clip(c_tmp1[:,0], c_min, c_max)
                c_tmp2[:,0] = np.clip(c_tmp2[:,0], c_min, c_max)
                c_tmp1[:,1] = np.clip(c_tmp1[:,1], width_min, width_max)
                c_tmp2[:,1] = np.clip(c_tmp2[:,1], width_min, width_max)
                denom = c_tmp1[m][l] - c_tmp2[m][l]
                if denom > 0:
                    D_J_c[m][l] += (OBJ(c_tmp1.tolist(),T,L,omega,H,GA,GB) - OBJ(c_tmp2.tolist(),T,L,omega,H,GA,GB)) / denom

        # Step 3, normalization for gradient
        norm2_D = 0
        for m in range(4):
            for l in range(L):
                norm2_D += D_J_c[m][l]**2
        norm_D = np.sqrt(norm2_D)
        if (not np.isfinite(norm_D)) or norm_D == 0:
            break
        D_J_c = [[eps*D_J_c[m][l]/norm_D for l in range(L)] for m in range(4)]

        # Step4, gradient ascend
        c_old = copy.deepcopy(c)
        c = np.array(c) + np.array(D_J_c)
        c[:,0] = np.clip(c[:,0], c_min, c_max)
        c[:,1] = np.clip(c[:,1], width_min, width_max)

        # Step5, Compute cost
        new_cost = OBJ(c.tolist(),T,L,omega,H,GA,GB)
        if new_cost > record_cost[-1]:
            print(new_cost)
            diff_cost = abs(new_cost - record_cost[-1])
            record_cost.append(new_cost)
        else:
            c = c_old
            dx_amp *= factor
            dx_width *= factor
            eps *= factor

    return record_cost, c.tolist(), count

# Example run. Comment this out if you only want the function definition.
fidelity_scaled, c_scaled, count_scaled = GRAPE_Gaussian_scaled(T,L,omega,H,GA,GB)
c = copy.deepcopy(c_scaled)
print('final scaled Gaussian-GRAPE fidelity:', fidelity_scaled[-1])
print('scaled Gaussian-GRAPE controls [amplitude, width(ns)]:')
print(np.array([[row[0], row[1] / ns] for row in c_scaled]))


In [ ]:
for i in range(2): 
    print('<'+str(i)+'>')
    fidelity,c, count = GRAPE(T,L,omega,H,GA,GB)
    print('------------')
    fidelity,c, count = GRAPE(T,L,omega,H,GA,GB,c)

In [ ]:
#record_c = []
#record_f = {}
#n_trails = 1
#start_time = time.time()
#for i in range(n_trails): # apply GRAPE with random initial multiple times
#    fidelity,c, count = GRAPE(T,L,omega,H,GA,GB)
#    record_f.update({i: fidelity})
#    record_c.append(c)
#    print(i) # check progress
#    plt.plot(fidelity)

#end_time = time.time()
#plt.legend([str(i) for i in range(n_trails)])
#plt.show()

In [ ]:
fidelity,c, count = GRAPE(T,L,omega,H,GA,GB,c)

In [ ]:
# Gaussian-parameter GRAPE-style optimizer
# Run cells 1-4 first so OBJ, T, L, omega, H, GA, and GB are defined.
# This does gradient ascent on the Gaussian ansatz: [amplitude, width] for C0, C1, C2, C3.
# Widths are represented in ns inside the optimizer so amplitude and width have comparable scale.

gaussian_grape_restarts = 3
gaussian_grape_maxiter = 150
gaussian_grape_learning_rate = 0.20
gaussian_grape_threshold = 1e-6
gaussian_grape_patience = 12
gaussian_grape_fd_step = np.array([0.05, 0.05] * 4, dtype=float)
gaussian_grape_width_min_ns = 0.05 * T / ns
gaussian_grape_width_max_ns = 1.50 * T / ns

gaussian_grape_bounds = []
for _ in range(4):
    gaussian_grape_bounds.append((c_min, c_max))
    gaussian_grape_bounds.append((gaussian_grape_width_min_ns, gaussian_grape_width_max_ns))

def gaussian_grape_pack(c_params):
    c_array = np.asarray(c_params, dtype=float).reshape(4, 2).copy()
    x = np.zeros(8, dtype=float)
    x[0::2] = c_array[:, 0]
    x[1::2] = c_array[:, 1] / ns
    return x

def gaussian_grape_unpack(x):
    x = np.asarray(x, dtype=float)
    c_array = np.zeros((4, 2), dtype=float)
    c_array[:, 0] = x[0::2]
    c_array[:, 1] = x[1::2] * ns
    return c_array.tolist()

def gaussian_grape_clip(x):
    x = np.asarray(x, dtype=float).copy()
    for idx, (lo, hi) in enumerate(gaussian_grape_bounds):
        x[idx] = np.clip(x[idx], lo, hi)
    return x

def gaussian_grape_initial_guess(seed_c=None):
    if seed_c is None:
        x = np.zeros(8, dtype=float)
        x[0::2] = np.random.uniform(c_min, c_max, 4)
        x[1::2] = np.random.uniform(0.20 * T / ns, 1.00 * T / ns, 4)
    else:
        x = gaussian_grape_pack(seed_c)
    return gaussian_grape_clip(x)

def gaussian_grape_fidelity_from_x(x):
    try:
        fidelity_value = OBJ(gaussian_grape_unpack(x), T, L, omega, H, GA, GB)
    except Exception:
        return -np.inf
    if not np.isfinite(fidelity_value):
        return -np.inf
    return float(np.real(fidelity_value))

def gaussian_grape_gradient(x):
    grad = np.zeros_like(x, dtype=float)
    for idx, step in enumerate(gaussian_grape_fd_step):
        lo, hi = gaussian_grape_bounds[idx]
        x_plus = x.copy()
        x_minus = x.copy()
        x_plus[idx] = min(hi, x[idx] + step)
        x_minus[idx] = max(lo, x[idx] - step)
        denom = x_plus[idx] - x_minus[idx]
        if denom <= 0:
            continue
        f_plus = gaussian_grape_fidelity_from_x(x_plus)
        f_minus = gaussian_grape_fidelity_from_x(x_minus)
        if np.isfinite(f_plus) and np.isfinite(f_minus):
            grad[idx] = (f_plus - f_minus) / denom
    return grad

def GRAPE_Gaussian(T, L, omega, H, GA, GB, initial_c=None):
    best_run = None

    for restart in range(gaussian_grape_restarts):
        seed_c = initial_c if restart == 0 and initial_c is not None else None
        x = gaussian_grape_initial_guess(seed_c)
        current_fidelity = gaussian_grape_fidelity_from_x(x)
        history = [current_fidelity]
        learning_rate = gaussian_grape_learning_rate
        no_improve = 0

        for iteration in range(gaussian_grape_maxiter):
            grad = gaussian_grape_gradient(x)
            grad_norm = np.linalg.norm(grad)
            if not np.isfinite(grad_norm) or grad_norm == 0:
                break

            direction = grad / grad_norm
            accepted = False
            trial_rate = learning_rate

            for _ in range(12):
                candidate_x = gaussian_grape_clip(x + trial_rate * direction)
                candidate_fidelity = gaussian_grape_fidelity_from_x(candidate_x)
                if candidate_fidelity > current_fidelity + gaussian_grape_threshold:
                    x = candidate_x
                    current_fidelity = candidate_fidelity
                    history.append(current_fidelity)
                    learning_rate = min(gaussian_grape_learning_rate, trial_rate * 1.1)
                    no_improve = 0
                    accepted = True
                    break
                trial_rate *= 0.5

            if not accepted:
                history.append(current_fidelity)
                learning_rate *= 0.5
                no_improve += 1

            if no_improve >= gaussian_grape_patience or learning_rate < 1e-5:
                break

        run = {
            'restart': restart,
            'x': x,
            'c': gaussian_grape_unpack(x),
            'fidelity': current_fidelity,
            'history': history,
        }
        print(f'restart {restart}: fidelity={current_fidelity:.6f}, steps={len(history)-1}')

        if best_run is None or run['fidelity'] > best_run['fidelity']:
            best_run = run

    return best_run['history'], best_run['c'], best_run

gaussian_grape_seed = c if 'c' in globals() else None
gaussian_grape_history, gaussian_grape_c, gaussian_grape_result = GRAPE_Gaussian(
    T, L, omega, H, GA, GB, gaussian_grape_seed
)
gaussian_grape_fidelity = gaussian_grape_history[-1]

# Store the optimized control in c so existing notebook cells can reuse it.
c = copy.deepcopy(gaussian_grape_c)

print('best Gaussian-GRAPE fidelity:', gaussian_grape_fidelity)
print('Gaussian-GRAPE controls [amplitude, width(ns)]:')
print(np.array([[row[0], row[1] / ns] for row in gaussian_grape_c]))

plt.figure(figsize=(7, 4))
plt.plot(gaussian_grape_history, marker='o')
plt.xlabel('Accepted step')
plt.ylabel('Fidelity')
plt.title('Gaussian-parameter GRAPE convergence')
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# Optimizes the 8 Gaussian parameters: amplitude and width for C0, C1, C2, and C3
from scipy.optimize import minimize

gaussian_opt_restarts = 10
gaussian_opt_maxiter = 500
gaussian_width_min = 0.05 * T
gaussian_width_max = 1.50 * T

gaussian_bounds = []
for _ in range(4):
    gaussian_bounds.append((c_min, c_max))
    gaussian_bounds.append((gaussian_width_min, gaussian_width_max))

def gaussian_unflatten(x):
    return np.asarray(x, dtype=float).reshape(4, 2).tolist()

def gaussian_initial_guess(seed_c=None):
    if seed_c is None:
        guess = np.zeros((4, 2), dtype=float)
        guess[:, 0] = np.random.uniform(c_min, c_max, 4)
        guess[:, 1] = np.random.uniform(0.20 * T, 1.00 * T, 4)
    else:
        guess = np.asarray(seed_c, dtype=float).reshape(4, 2).copy()

    guess[:, 0] = np.clip(guess[:, 0], c_min, c_max)
    guess[:, 1] = np.where(np.isfinite(guess[:, 1]), guess[:, 1], 0.50 * T)
    guess[:, 1] = np.clip(guess[:, 1], gaussian_width_min, gaussian_width_max)
    return guess.reshape(-1)

def gaussian_negative_fidelity(x):
    fidelity_value = OBJ(gaussian_unflatten(x), T, L, omega, H, GA, GB)
    if not np.isfinite(fidelity_value):
        return 1e6
    return -float(np.real(fidelity_value))

best_gaussian_result = None

for restart in range(gaussian_opt_restarts):
    seed_c = c if restart == 0 and 'c' in globals() else None
    x0 = gaussian_initial_guess(seed_c)

    result = minimize(
        gaussian_negative_fidelity,
        x0,
        method='L-BFGS-B',
        bounds=gaussian_bounds,
        options={'maxiter': gaussian_opt_maxiter, 'ftol': 1e-8, 'maxls': 20},
    )

    restart_fidelity = -float(result.fun)
    print(f'restart {restart}: fidelity={restart_fidelity:.6f}, success={result.success}')

    if best_gaussian_result is None or result.fun < best_gaussian_result.fun:
        best_gaussian_result = result

gaussian_opt_result = best_gaussian_result
gaussian_opt_c = gaussian_unflatten(gaussian_opt_result.x)
gaussian_opt_fidelity = OBJ(gaussian_opt_c, T, L, omega, H, GA, GB)

# Store the optimized control in c so existing notebook cells can reuse it
c = copy.deepcopy(gaussian_opt_c)

print('best fidelity:', gaussian_opt_fidelity)
print('optimized Gaussian controls [amplitude, width(ns)]:')
print(np.array([[row[0], row[1] / ns] for row in gaussian_opt_c]))


In [ ]:
# Local 3D landscape around the optimized Gaussian solution
# Varies one channel's amplitude and width while keeping all other parameters fixed

if 'gaussian_opt_c' not in globals():
    if 'c' not in globals():
        raise ValueError('Run the Gaussian optimizer cell first, or define c before this cell.')
    gaussian_opt_c = copy.deepcopy(c)
    gaussian_opt_fidelity = OBJ(gaussian_opt_c, T, L, omega, H, GA, GB)

landscape_channel = 0  # 0=C0, 1=C1, 2=C2, 3=C3
local_grid_size = 31
local_width_floor = globals().get('gaussian_width_min', 0.05 * T)
local_width_ceiling = globals().get('gaussian_width_max', 1.50 * T)

base_c = copy.deepcopy(gaussian_opt_c)
center_amplitude = float(base_c[landscape_channel][0])
center_width = float(base_c[landscape_channel][1])

amplitude_span = 0.25 * (c_max - c_min)
amplitude_min = max(c_min, center_amplitude - amplitude_span)
amplitude_max = min(c_max, center_amplitude + amplitude_span)

width_min = max(local_width_floor, 0.50 * center_width)
width_max = min(local_width_ceiling, 1.50 * center_width)
if width_min >= width_max:
    width_min = local_width_floor
    width_max = local_width_ceiling

amplitude_values = np.linspace(amplitude_min, amplitude_max, local_grid_size)
width_values = np.linspace(width_min, width_max, local_grid_size)
A_grid, W_grid = np.meshgrid(amplitude_values, width_values)
F_grid = np.zeros_like(A_grid, dtype=float)

for row, width in enumerate(width_values):
    for col, amplitude in enumerate(amplitude_values):
        c_trial = copy.deepcopy(base_c)
        c_trial[landscape_channel][0] = float(amplitude)
        c_trial[landscape_channel][1] = float(width)
        F_grid[row, col] = OBJ(c_trial, T, L, omega, H, GA, GB)

best_row, best_col = np.unravel_index(np.argmax(F_grid), F_grid.shape)
best_amplitude = A_grid[best_row, best_col]
best_width = W_grid[best_row, best_col]
best_fidelity = F_grid[best_row, best_col]

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
surface = ax.plot_surface(
    A_grid,
    W_grid / ns,
    F_grid,
    cmap='viridis',
    edgecolor='none',
    alpha=0.9,
)
ax.scatter(
    [center_amplitude],
    [center_width / ns],
    [gaussian_opt_fidelity],
    color='red',
    s=80,
    label='optimized point',
)
ax.scatter(
    [best_amplitude],
    [best_width / ns],
    [best_fidelity],
    color='black',
    s=55,
    label='best grid point',
)
ax.set_xlabel(f'C{landscape_channel} Gaussian amplitude')
ax.set_ylabel(f'C{landscape_channel} Gaussian width (ns)')
ax.set_zlabel('Fidelity')
ax.set_title(f'Local Gaussian landscape around optimized C{landscape_channel}; max={best_fidelity:.4f}')
fig.colorbar(surface, ax=ax, shrink=0.65, pad=0.1, label='Fidelity')
ax.legend()
plt.show()

print('optimized fidelity:', gaussian_opt_fidelity)
print('optimized amplitude:', center_amplitude)
print('optimized width (ns):', center_width / ns)
print('best grid fidelity:', best_fidelity)
print('best grid amplitude:', best_amplitude)
print('best grid width (ns):', best_width / ns)
